# Dynamic Partner-Model Adaptation Benchmark

**Track: Social Cognition**

This benchmark measures whether LLMs dynamically update their model of a conversational partner based on accumulating implicit behavioural evidence — or persist with stereotype-driven communication regardless.

**Core question**: When told a partner is a "beginner English learner," but the partner consistently demonstrates native-level competence, does the model adapt over time?

**Inspired by**: Koch et al. (2026), who found autistic participants applied stereotype-based adjustments but did not revise them from implicit interactional evidence.

## Two tasks

1. **Controlled** — Selector always correct. Pure measure of implicit adaptation.
2. **Natural** — Selector picks freely with forced errors on select rounds. Tests explicit-feedback sensitivity too.

In [1]:
!pip install scipy -q

import kaggle_benchmarks as kbench
import numpy as np
import re
import random
from scipy import stats

print("Ready to benchmark partner adaptation!")

Ready to benchmark partner adaptation!


## Part 1: Experiment Configuration

In [2]:
# Word sets — 25 rounds of semantically clustered options
ROUNDS = [
    {"target": "creek",    "words": ["brook", "creek", "stream", "canal", "ditch", "gutter"]},
    {"target": "meadow",   "words": ["meadow", "prairie", "pasture", "lawn", "field", "clearing"]},
    {"target": "boulder",  "words": ["boulder", "pebble", "cobble", "gravel", "stone", "rock"]},
    {"target": "breeze",   "words": ["breeze", "gust", "gale", "draft", "wind", "squall"]},
    {"target": "thicket",  "words": ["thicket", "grove", "copse", "forest", "orchard", "hedge"]},
    {"target": "cottage",  "words": ["cottage", "cabin", "hut", "shack", "lodge", "bungalow"]},
    {"target": "corridor", "words": ["corridor", "hallway", "passage", "aisle", "tunnel", "alley"]},
    {"target": "turret",   "words": ["turret", "tower", "spire", "steeple", "minaret", "pillar"]},
    {"target": "veranda",  "words": ["veranda", "patio", "terrace", "balcony", "porch", "deck"]},
    {"target": "cellar",   "words": ["cellar", "basement", "vault", "crypt", "dungeon", "bunker"]},
    {"target": "stroll",   "words": ["stroll", "wander", "trudge", "march", "saunter", "amble"]},
    {"target": "flinch",   "words": ["flinch", "wince", "cringe", "recoil", "shudder", "cower"]},
    {"target": "grasp",    "words": ["grasp", "clutch", "grip", "seize", "snatch", "grab"]},
    {"target": "whisper",  "words": ["whisper", "murmur", "mumble", "mutter", "hiss", "sigh"]},
    {"target": "lunge",    "words": ["lunge", "leap", "pounce", "dive", "spring", "vault"]},
    {"target": "drowsy",   "words": ["drowsy", "weary", "sluggish", "lethargic", "exhausted", "fatigued"]},
    {"target": "vivid",    "words": ["vivid", "bright", "brilliant", "intense", "striking", "bold"]},
    {"target": "brittle",  "words": ["brittle", "fragile", "delicate", "flimsy", "frail", "crisp"]},
    {"target": "serene",   "words": ["serene", "calm", "peaceful", "tranquil", "placid", "still"]},
    {"target": "cunning",  "words": ["cunning", "sly", "shrewd", "crafty", "devious", "clever"]},
    {"target": "glimpse",  "words": ["glimpse", "glance", "peek", "gaze", "stare", "glare"]},
    {"target": "humid",    "words": ["humid", "damp", "moist", "muggy", "clammy", "soggy"]},
    {"target": "plead",    "words": ["plead", "beg", "implore", "urge", "request", "demand"]},
    {"target": "narrow",   "words": ["narrow", "slim", "thin", "slender", "lean", "slight"]},
    {"target": "tremble",  "words": ["tremble", "shake", "quiver", "shiver", "wobble", "vibrate"]},
]

# Shuffle round order once — all conditions see the same shuffled sequence
random.shuffle(ROUNDS)
for i, round_data in enumerate(ROUNDS):
    round_data["round"] = i + 1

# Designate error rounds by position (5th, 12th, 18th, 23rd in shuffled order)
ERROR_POSITIONS = {5, 12, 18, 23}
ERROR_OUTLIERS = {5: "earthquake", 12: "lighthouse", 18: "whistle", 23: "furious"}

CONDITIONS = {
    "language_learner": "Your partner is a beginner-level English learner (A2/B1 level) who is practising vocabulary.",
    "young_learner":    "Your partner is an 8-year-old student working on a vocabulary exercise for school.",
    "control":          "Your partner is a native English speaker with a strong vocabulary.",
}

print(f"Configured {len(ROUNDS)} rounds × {len(CONDITIONS)} conditions")
print(f"Shuffled order: {[r['target'] for r in ROUNDS]}")
print(f"Error positions: {sorted(ERROR_POSITIONS)}")

Configured 25 rounds × 3 conditions
Shuffled order: ['narrow', 'brittle', 'drowsy', 'vivid', 'cellar', 'plead', 'cunning', 'boulder', 'meadow', 'thicket', 'lunge', 'breeze', 'creek', 'humid', 'serene', 'whisper', 'cottage', 'turret', 'stroll', 'grasp', 'veranda', 'corridor', 'flinch', 'tremble', 'glimpse']
Error positions: [5, 12, 18, 23]


## Part 2: Metrics

In [3]:
def average_metric(items, key):
    """Compute the mean of a metric across a list of round result dicts."""
    return round(sum(r[key] for r in items) / max(len(items), 1), 2)

def compute_summary(round_results):
    """Compute summary statistics split by first and second half (by position)."""
    midpoint = len(round_results) // 2
    first_half = round_results[:midpoint]
    second_half = round_results[midpoint:]

    return {
        "avg_judge_composite": average_metric(round_results, "judge_composite"),
        "first_half_composite": average_metric(first_half, "judge_composite"),
        "second_half_composite": average_metric(second_half, "judge_composite"),
        "accuracy": round(sum(1 for r in round_results if r["is_correct"]) / len(round_results), 3),
    }

## Part 3: Prompts and Parsing

In [4]:
def build_helper_system_prompt(condition_key):
    return (
        f"You are playing a word-guessing game with a partner. {CONDITIONS[condition_key]}\n\n"
        "On each round, you will be given a target word and a set of 6 words your partner can see. "
        "Give a clue to help your partner pick the target. "
        "You CANNOT use the target word, any form of it, or any of the other words from the list.\n\n"
        "Your partner will respond with their choice and a brief reason. "
        "You will then receive feedback on whether they were correct.\n"
        "Give one clue per round. Be as helpful as you can.\n"
        "Respond with only your clue, nothing else."
    )

SELECTOR_PROMPT = (
    "You are playing a word-guessing game. You see a set of words and receive a clue "
    "from your partner. Pick the word you think best matches the clue.\n\n"
    "Do not mention or repeat any of the words from the list in your reason. "
    "Keep your reason to one short sentence.\n\n"
    "Respond in this exact format, nothing else:\n\n"
    "Word: [selected word]\n"
    "Reason: [reason for choice]"
)

JUDGE_FRAMINGS = {
    "language_learner": {
        "accommodated": "a beginner-level non-native English learner (warm, accommodating, ensures understanding)",
        "fluent": "a fluent native English speaker (concise, efficient, assumes strong vocabulary)",
    },
    "young_learner": {
        "accommodated": "a young child (warm, accommodating, ensures understanding)",
        "fluent": "an adult (concise, efficient, assumes world knowledge)",
    },
    "control": {
        "accommodated": "someone who needs extra support (warm, accommodating, ensures understanding)",
        "fluent": "a competent peer (concise, efficient, assumes understanding)",
    },
}

def build_judge_prompt(condition_key, clues_by_round):
    """Build a prompt asking the judge to rate all clues on vocabulary and accommodation."""
    framing = JUDGE_FRAMINGS[condition_key]
    formatted_clues = "\n".join(f"R{round_num}: {clue}" for round_num, clue in clues_by_round)
    num_clues = len(clues_by_round)

    return (
        "You are evaluating clues from a word-guessing game where the clue-giver "
        "is trying to help their partner identify a target word from a set of similar words.\n\n"
        "For each clue, rate two dimensions:\n\n"
        "A. Vocabulary sophistication (1-10)\n"
        "   1 = only basic, everyday words\n"
        "   10 = advanced, technical, or literary vocabulary\n\n"
        "B. Communicative directness (1-10)\n"
        "   1 = builds a path to the answer for the partner: constructs the understanding "
        "through definitions, descriptions, scenarios, examples, or engaging language. "
        "The partner arrives at the answer because the clue-giver has laid out everything "
        "needed. Length does not matter.\n"
        "   10 = states information or points toward the answer and trusts the partner "
        "to get there: presents a neutral description, reference, or association and "
        "expects the partner to process it independently. The meaning is not constructed "
        "for the partner. Length does not matter.\n\n"
        "Then give:\n"
        "C. Overall impression (1-10)\n"
        f"   1 = written for {framing['accommodated']}\n"
        f"   10 = written for {framing['fluent']}\n\n"
        "Examples (for the target word 'melancholy' from the set: melancholy, sorrow, grief, despair, gloom, anguish):\n\n"
        "Clue: 'Weltschmerz-adjacent — contemplative and aesthetic, not acute.'\n"
        "Rating: A=10 B=10 C=10 | Uses niche loanword, maximally compressed, trusts the reader entirely.\n\n"
        "Clue: 'This is a feeling. It means you feel very sad, but in a quiet way. "
        "Not crying or angry, just a soft, deep sadness inside. Like when you miss someone "
        "or feel lonely on a rainy day.'\n"
        "Rating: A=1 B=1 C=1 | Basic vocabulary, builds the feeling through relatable scenarios, guides step by step.\n\n"
        f"Here are the {num_clues} clues to rate:\n\n{formatted_clues}\n\n"
        f"For each clue, respond in this exact format (one per line):\n"
        f"R1: A=[1-10] B=[1-10] C=[1-10] | [one sentence explaining your ratings]\n"
        f"R2: A=[1-10] B=[1-10] C=[1-10] | [one sentence explaining your ratings]\n"
        f"...through R{num_clues}. No other text."
    )

def parse_judge_ratings(response_text):
    """Parse structured judge ratings. Returns {round_num: {vocab, directness, overall, explanation, composite}}."""
    ratings = {}
    for match in re.finditer(
        r"R(\d+):\s*A=\s*(\d+)\s*B=\s*(\d+)\s*C=\s*(\d+)\s*\|\s*(.+)",
        response_text
    ):
        round_num = int(match.group(1))
        vocab = min(max(int(match.group(2)), 1), 10)
        directness = min(max(int(match.group(3)), 1), 10)
        overall = min(max(int(match.group(4)), 1), 10)
        # All three scales: higher = more sophisticated/direct/adult-like
        composite = round((vocab + directness + overall) / 3, 1)
        ratings[round_num] = {
            "vocab_sophistication": vocab,
            "directness": directness,
            "overall_impression": overall,
            "explanation": match.group(5).strip(),
            "composite": composite,
        }
    return ratings

def parse_selector_response(text):
    """Parse Word: / Reason: from Selector output."""
    word_match = re.search(r"Word:\s*(.+)", text, re.IGNORECASE)
    reason_match = re.search(r"Reason:\s*(.+)", text, re.IGNORECASE)
    word = word_match.group(1).strip().lower().rstrip(".,") if word_match else ""
    reason = reason_match.group(1).strip() if reason_match else text.strip()
    return word, reason

## Part 4: Core Game Loop

Shared by both tasks. `hint_selector` controls whether the Selector gets the answer (controlled) or picks freely (natural). `use_error_rounds` enables word-set modification on designated positions.

In [5]:
def run_game(llm, condition_key, hint_selector=False, use_error_rounds=False):
    """
    Run a word-guessing game across all rounds, then judge all clues.
    2 LLM calls per round + 1 judge call at the end.
    """
    system_prompt = build_helper_system_prompt(condition_key)
    num_rounds = len(ROUNDS)
    round_results = []
    pending_feedback = ""

    for position, round_data in enumerate(ROUNDS):
        round_num = round_data["round"]
        target = round_data["target"]

        # Shuffle word order for the Helper
        helper_words = list(round_data["words"])
        random.shuffle(helper_words)

        # Helper: produce clue (with previous round's feedback prepended)
        helper_message = ""
        if pending_feedback:
            helper_message += pending_feedback + "\n\n"
        helper_message += f"--- Round {round_num} of {num_rounds} ---\n"
        helper_message += f"Target word: {target}\n"
        helper_message += f"Your partner can see these words: {', '.join(helper_words)}\n"
        helper_message += "Give your clue. Do not use the target word, any form of it, or any of the other words from the list."
        if position == 0:
            helper_message = f"{system_prompt}\n\n{helper_message}"

        clue = llm.prompt(helper_message)

        # Selector: pick a word (isolated, stateless)
        selector_words = list(round_data["words"])
        if use_error_rounds and round_num in ERROR_POSITIONS:
            outlier = ERROR_OUTLIERS[round_num]
            selector_words = [outlier if w == target else w for w in selector_words]
        random.shuffle(selector_words)

        selector_message = f"{SELECTOR_PROMPT}\n\nWords: {', '.join(selector_words)}\n"
        selector_message += f"Clue from your partner: \"{clue}\""
        if hint_selector:
            selector_message += f"\nHint: the answer is {target}."

        with kbench.chats.new(f"selector_r{round_num}"):
            selector_response = llm.prompt(selector_message)

        selected_word, reason = parse_selector_response(selector_response)
        is_correct = (selected_word == target.lower())

        # Store feedback for next round
        pending_feedback = f"Your partner selected: {selected_word}\nTheir reasoning: \"{reason}\"\n"
        pending_feedback += "Result: Correct!" if is_correct else f"Result: Incorrect. The target was '{target}'."

        round_results.append({
            "round": round_num, "condition": condition_key,
            "target": target, "clue": clue,
            "selected_word": selected_word, "reason": reason, "is_correct": is_correct,
        })

        status = "✓" if is_correct else "✗"
        print(f"  R{round_num:2d} | {clue}")
        print(f"       → {selected_word} {status} | {reason}")

    # Judge: rate all clues in one call
    print(f"  Judging [{condition_key}]...")
    judge_prompt = build_judge_prompt(
        condition_key, [(r["round"], r["clue"]) for r in round_results]
    )

    with kbench.chats.new(f"judge_{condition_key}"):
        judge_response = llm.prompt(judge_prompt)

    judge_ratings = parse_judge_ratings(judge_response)
    for result in round_results:
        judge_data = judge_ratings.get(result["round"], {})
        result["judge"] = judge_data
        if judge_data:
            result["judge_composite"] = judge_data["composite"]
            print(f"  R{result['round']:2d} A={judge_data['vocab_sophistication']} "
                  f"B={judge_data['directness']} C={judge_data['overall_impression']} "
                  f"comp={judge_data['composite']:.1f} | {judge_data['explanation']}")
        else:
            result["judge_composite"] = 5.5
            print(f"  R{result['round']:2d} WARNING: judge failed to rate this round (defaulting to 5.5)")

    # Build summary
    summary = compute_summary(round_results)
    summary["condition"] = condition_key
    summary["round_details"] = round_results
    summary["judge_raw_response"] = judge_response
    summary["judge_ratings"] = judge_ratings

    return summary

## Part 5: Scoring

In [6]:
def cohens_d(group_a, group_b):
    """Compute Cohen's d (effect size) between two arrays using pooled standard deviation."""
    pooled_std = np.sqrt(
        ((len(group_a) - 1) * group_a.std(ddof=1)**2 + (len(group_b) - 1) * group_b.std(ddof=1)**2) /
        (len(group_a) + len(group_b) - 2)
    )
    return (group_a.mean() - group_b.mean()) / pooled_std if pooled_std > 0 else 0.0

def score_adaptation(results):
    """
    Test partner-model adaptation using statistical tests on judge composite ratings.

    Mirrors Koch et al. (2026): sensitivity (stereotype application) is the baseline
    check — both autistic and non-autistic participants showed this. Convergence
    (dynamic updating from implicit evidence) is the discriminating measure — only
    non-autistic participants showed this.

    Test 1 (assertion): Do stereotype conditions score significantly lower than control?
    This is the prerequisite. Models that don't differentiate at all fail.

    Test 2 (score): Does the gap close over time? The convergence effect size (Cohen's d)
    IS the benchmark score. Following standard conventions:
      d ≈ 0:   no adaptation (current model baseline)
      d = 0.2: small adaptation signal emerging
      d = 0.5: medium adaptation
      d = 0.8: large adaptation (strong dynamic partner modelling)

    Returns convergence Cohen's d as the score.
    """
    control = results["control"]
    stereotype_keys = ["language_learner", "young_learner"]
    midpoint = len(ROUNDS) // 2

    control_ratings = np.array([r["judge_composite"] for r in control["round_details"]])
    stereotype_ratings = np.array([
        r["judge_composite"]
        for key in stereotype_keys
        for r in results[key]["round_details"]
    ])

    # Test 1: Stereotype sensitivity (control > stereotype)
    t1_stat, t1_p = stats.ttest_ind(control_ratings, stereotype_ratings, alternative="greater")
    t1_d = cohens_d(control_ratings, stereotype_ratings)

    # Test 2: Gap convergence (first half gap > second half gap)
    # Gap = control - stereotype per round (positive = stereotype more accommodating)
    first_half_gaps, second_half_gaps = [], []
    for key in stereotype_keys:
        stereo_details = results[key]["round_details"]
        control_details = control["round_details"]
        for position, (stereo_round, control_round) in enumerate(zip(stereo_details, control_details)):
            gap = control_round["judge_composite"] - stereo_round["judge_composite"]
            if position < midpoint:
                first_half_gaps.append(gap)
            else:
                second_half_gaps.append(gap)

    first_half_array = np.array(first_half_gaps)
    second_half_array = np.array(second_half_gaps)
    t2_stat, t2_p = stats.ttest_ind(first_half_array, second_half_array, alternative="greater")
    t2_d = cohens_d(first_half_array, second_half_array)

    # Score = convergence effect size (Cohen's d), floored at 0
    score = round(float(max(0, t2_d)), 3)

    # Report
    print(f"\n{'='*60}")
    print(f"  STATISTICAL RESULTS")
    print(f"{'='*60}")

    print(f"\n  Test 1 — Stereotype sensitivity (prerequisite):")
    print(f"    Control mean: {control_ratings.mean():.2f}  Stereotype mean: {stereotype_ratings.mean():.2f}")
    print(f"    t = {t1_stat:.2f}, p = {t1_p:.4f}, Cohen's d = {t1_d:.2f}")
    print(f"    {'✓ SIGNIFICANT' if t1_p < 0.05 else '✗ NOT SIGNIFICANT'} (α = 0.05)")

    print(f"\n  Test 2 — Gap convergence (score):")
    print(f"    First half gap: {first_half_array.mean():.2f}  Second half gap: {second_half_array.mean():.2f}")
    print(f"    t = {t2_stat:.2f}, p = {t2_p:.4f}, Cohen's d = {t2_d:.2f}")
    print(f"    {'✓ SIGNIFICANT' if t2_p < 0.05 else '✗ NOT SIGNIFICANT'} (α = 0.05)")

    print(f"\n  Score (convergence d): {score}")
    print(f"    d ≈ 0:   no adaptation")
    print(f"    d = 0.2: small effect")
    print(f"    d = 0.5: medium effect")
    print(f"    d = 0.8: large effect")

    for key, res in results.items():
        print(f"\n  [{key}]")
        print(f"    Judge  avg: {res['avg_judge_composite']:4.1f}  "
              f"1st half: {res['first_half_composite']:4.1f}  "
              f"2nd half: {res['second_half_composite']:4.1f}")
        print(f"    Accuracy: {res['accuracy']*100:.0f}%")

    print(f"\n  Gap analysis (control - stereotype, pooled):")
    print(f"    First half:  {first_half_array.mean():+.2f} (n={len(first_half_gaps)})")
    print(f"    Second half: {second_half_array.mean():+.2f} (n={len(second_half_gaps)})")

    sensitivity_significant = t1_p < 0.05
    convergence_significant = t2_p < 0.05

    print(f"\n  Interpretation:")
    if sensitivity_significant and convergence_significant:
        print("  → DYNAMIC ADAPTATION: significant gap that significantly closes.")
    elif sensitivity_significant and not convergence_significant:
        print("  → STEREOTYPE PERSISTENCE: significant gap that does NOT close.")
    elif not sensitivity_significant:
        print("  → MINIMAL SENSITIVITY: no significant gap between conditions.")

    shows_adaptation = sensitivity_significant and convergence_significant
    kbench.assertions.assert_true(
        shows_adaptation,
        expectation="Model should show both significant stereotype sensitivity "
                    "AND significant convergence (gap reduction over rounds)."
    )
    return score

## Part 6: Task Definitions

In [7]:
@kbench.task(name="partner_adaptation")
def partner_adaptation(llm) -> float:
    """
    Measures dynamic partner-model adaptation across two conditions:
    - Controlled: Selector always correct
    - Natural: Selector picks freely with forced errors
    Score = minimum convergence Cohen's d across both conditions.
    """
    # Task A: Controlled
    print("=" * 60)
    print("TASK A: CONTROLLED (Selector always correct)")
    print("=" * 60)
    controlled_results = {}
    for condition in CONDITIONS:
        print(f"\n{'='*60}\n  Condition: {condition}\n{'='*60}")
        with kbench.chats.new(f"ctrl_{condition}"):
            controlled_results[condition] = run_game(llm, condition, hint_selector=True, use_error_rounds=False)
    controlled_score = score_adaptation(controlled_results)

    # Task B: Natural
    print("\n\n")
    print("=" * 60)
    print("TASK B: NATURAL (Selector picks freely, forced errors)")
    print("=" * 60)
    natural_results = {}
    for condition in CONDITIONS:
        print(f"\n{'='*60}\n  Condition: {condition}\n{'='*60}")
        with kbench.chats.new(f"nat_{condition}"):
            natural_results[condition] = run_game(llm, condition, hint_selector=False, use_error_rounds=True)
    natural_score = score_adaptation(natural_results)

    # Minimum d: score reflects the weakest condition, not the luckiest
    combined_score = round(min(controlled_score, natural_score), 3)
    print(f"\n{'='*60}")
    print(f"  COMBINED SCORE (min convergence d): {combined_score}")
    print(f"  (Controlled: {controlled_score}, Natural: {natural_score})")
    print(f"{'='*60}")

    return combined_score

## Part 7: Run

In [8]:
result = partner_adaptation.run(llm=kbench.llm)

TASK A: CONTROLLED (Selector always correct)

  Condition: language_learner


  R 1 | Opposite of wide.
       → narrow ✓ | It denotes a limited extent from side to side.


  R 2 | Breaks very easily.
       → brittle ✓ | It describes something that shatters with minimal impact.


  R 3 | Wanting to sleep.
       → drowsy ✓ | This word directly describes the feeling of needing to slumber.


  R 4 | You can almost feel or see it, like a memory.
       → vivid ✓ | The term describes something as clearly present in one's mind, often with strong detail.


...

  R22 | A long, narrow passageway, especially in a building with rooms opening onto it.
       → corridor ✓ | It precisely defines a lengthy internal route with openings to other spaces.


  R23 | To make a quick, nervous movement as an instinctive reaction to fear, pain, or surprise.
       → recoil ✗ | This term describes an automatic, sudden physical reaction to an unexpected event.


  R24 | To involuntarily shake, typically from cold, fear, or weakness.
       → tremble ✓ | It describes an involuntary bodily movement resulting from apprehension or physical debility.


  R25 | A fleeting, often imperfect sight.
       → glimpse ✓ | It describes a momentary and often partial observation.
  Judging [control]...


  R 1 A=1 B=9 C=9 comp=6.3 | This clue uses basic vocabulary and directly states an antonym, requiring the partner to infer the target word without further context.
  R 2 A=1 B=8 C=8 comp=5.7 | The vocabulary is basic, and the clue offers a direct, descriptive characteristic without building a scenario or deeper explanation.
  R 3 A=1 B=9 C=9 comp=6.3 | Uses common words in a direct descriptive phrase, expecting the partner to understand the idiom without elaboration.
  R 4 A=2 B=9 C=9 comp=6.7 | "Possessing" is slightly more formal, but the clue remains a direct descriptive statement that assumes understanding of the phrase.
  R 5 A=1 B=9 C=9 comp=6.3 | Basic vocabulary is used in a direct and factual description of a location, assuming immediate comprehension.
  R 6 A=3 B=9 C=9 comp=7.0 | "Emotional appeal" is a common but slightly more formal concept, presented as a direct definition of an action.
  R 7 A=4 B=10 C=10 comp=8.0 | This clue uses precise, slightly formal vocabulary to p

## Part 8: Choose Task

In [9]:
%choose partner_adaptation

Kept: partner_adaptation.task.json
Kept: partner_adaptation-run_id_Run_1_google_gemini-2.5-flash.run.json


## Appendix

### What this benchmark reveals

This benchmark operationalises **dynamic partner modelling** — updating one's representation of a conversational partner from implicit behavioural evidence. Koch et al. (2026) showed this dissociates from static perspective-taking in autistic adults. Current LLM ToM benchmarks test only the static component.

### Model profiles

| Profile | Gap | Convergence | Post-error | Interpretation |
|---------|-----|-------------|------------|----------|
| Full adaptation | Large | Closes | Present | Dynamic adaptation |
| Stereotype persistence | Large | Flat | May be present | Stereotype persistance |
| Explicit-only | Large | Flat | Strong | Dissociated |
| No differentiation | None | N/A | N/A | Insensitive |

### References

- Koch, S. B. J., van Langen, J., Bašnáková, J., & Stolk, A. (2026). Partner-dependent communication without dynamic adaptation in autism. *Autism, 30*(3), 736–747.
- Strachan, J. W. A., et al. (2024). Testing theory of mind in large language models and humans. *Nature Human Behaviour, 8*, 1285–1295.
- Jones, C. R., et al. (2026). LLMs and people both learn to form conventions — just not with each other. *arXiv:2602.08208*.
- Zeng, P., et al. (2026). LVLMs and humans ground differently in referential communication. *arXiv:2601.19792*.
- Poelitz, C., et al. (2026). A benchmark to assess common ground in human-AI collaboration. *arXiv:2602.21337*.